# 61 — Report Catalog Ingest

Ingests the documentary evidence layer from `data_sources/` and writes a
traceable inventory and preview for downstream notebooks.

In [ ]:
import os, json, hashlib, platform, sys, re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        if path.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(path), "status": "empty_file"}
        df = pd.read_csv(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def safe_read_parquet(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        df = pd.read_parquet(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest: dict, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def standardise_pollutant(x):
    s = str(x).strip().lower()
    mapping = {
        "oxides of nitrogen": "NOx",
        "nox": "NOx",
        "no2": "NO2",
        "no": "NO",
        "particulates": "Particulates",
        "dust": "Particulates",
        "pm10": "PM10",
        "pm2.5": "PM2.5",
        "sulphur dioxide": "SO2",
        "so2": "SO2",
        "hydrogen chloride": "HCl",
        "hcl": "HCl",
        "total organic carbon": "TOC",
        "toc": "TOC",
        "carbon monoxide": "CO",
        "co": "CO",
        "ammonia": "NH3",
        "nh3": "NH3",
    }
    return mapping.get(s, str(x).strip())

In [ ]:
step = "61_report_catalog_ingest"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

files = {
    "catalog_csv": DATA / "incinerator_report_catalog.csv",
    "catalog_parquet": DATA / "incinerator_report_catalog.parquet",
    "blocks_csv": DATA / "incinerator_report_extracted_blocks.csv",
    "blocks_parquet": DATA / "incinerator_report_extracted_blocks.parquet",
    "cems_vocab": DATA / "cems_vocab_hits.csv",
    "cems_blocks": DATA / "cems_table_blocks_hits.csv",
    "emissions_long": DATA / "emissions_long.parquet",
}

cat_csv, m1 = safe_read_csv(files["catalog_csv"])
cat_pq, m2 = safe_read_parquet(files["catalog_parquet"])
blocks_csv, m3 = safe_read_csv(files["blocks_csv"])
blocks_pq, m4 = safe_read_parquet(files["blocks_parquet"])
cems_vocab, m5 = safe_read_csv(files["cems_vocab"])
cems_blocks, m6 = safe_read_csv(files["cems_blocks"])
emissions, m7 = safe_read_parquet(files["emissions_long"])

catalog = cat_csv if not cat_csv.empty else cat_pq
blocks = blocks_csv if not blocks_csv.empty else blocks_pq

preview = catalog.head(50).copy() if not catalog.empty else pd.DataFrame()
preview.to_csv(out / "catalog_preview.csv", index=False)

summary = {
    "catalog_rows": int(len(catalog)),
    "blocks_rows": int(len(blocks)),
    "cems_vocab_rows": int(len(cems_vocab)),
    "cems_blocks_rows": int(len(cems_blocks)),
    "emissions_rows": int(len(emissions)),
    "inputs_present": {
        "catalog": not catalog.empty,
        "blocks": not blocks.empty,
        "cems_vocab": not cems_vocab.empty,
        "cems_blocks": not cems_blocks.empty,
        "emissions": not emissions.empty,
    }
}
write_json(out / "documentary_ingest_summary.json", summary)

manifest = manifest_base(step, [CONFIGS / "documentary_sources.yml"])
manifest["inputs"] = {
    "catalog_csv": m1, "catalog_parquet": m2, "blocks_csv": m3, "blocks_parquet": m4,
    "cems_vocab": m5, "cems_blocks": m6, "emissions_long": m7
}
add_artifact(manifest, out / "catalog_preview.csv")
add_artifact(manifest, out / "documentary_ingest_summary.json")
write_json(out / "manifest.json", manifest)

print(summary)
display(preview.head(20))